In [2]:
# ==============================
# TASK 3: A/B HYPOTHESIS TESTING
# ==============================

import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency, ttest_ind

# Load dataset
df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")

# Create required metrics
df["ClaimOccurred"] = df["TotalClaims"] > 0
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# Avoid division by zero
df["LossRatio"] = np.where(
    df["TotalPremium"] != 0,
    df["TotalClaims"] / df["TotalPremium"],
    np.nan
)

df.head()
# ==============================
# HELPER FUNCTIONS
# ==============================

def chi_square_test(data, group_col, target_col):
    contingency_table = pd.crosstab(
        data[group_col],
        data[target_col]
    )

    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    return p_value


def t_test_between_groups(data, group_col, value_col, group_a, group_b):
    group_a_values = data[data[group_col] == group_a][value_col].dropna()
    group_b_values = data[data[group_col] == group_b][value_col].dropna()

    t_stat, p_value = ttest_ind(
        group_a_values,
        group_b_values,
        equal_var=False
    )

    return p_value


def decision(p_value, alpha=0.05):
    if p_value < alpha:
        return "Reject H0"
    return "Fail to Reject H0"
# ==============================
# HYPOTHESIS 1
# H0: No risk difference across provinces
# KPI: Claim Frequency
# ==============================

province_p_value = chi_square_test(
    data=df,
    group_col="Province",
    target_col="ClaimOccurred"
)

print("Hypothesis 1 - Provinces")
print("P-value:", province_p_value)
print("Decision:", decision(province_p_value))


# ==============================
# HYPOTHESIS 2
# H0: No risk difference between zip codes
# KPI: Claim Frequency
# ==============================

# Select top 2 most common postal codes
top_postal_codes = (
    df["PostalCode"]
    .value_counts()
    .head(2)
    .index
)

postal_df = df[
    df["PostalCode"].isin(top_postal_codes)
]

postal_p_value = chi_square_test(
    data=postal_df,
    group_col="PostalCode",
    target_col="ClaimOccurred"
)

print("\nHypothesis 2 - Postal Codes")
print("P-value:", postal_p_value)
print("Decision:", decision(postal_p_value))


# ==============================
# HYPOTHESIS 3
# H0: No significant margin difference between zip codes
# KPI: Margin
# ==============================

postal_code_a = top_postal_codes[0]
postal_code_b = top_postal_codes[1]

margin_p_value = t_test_between_groups(
    data=postal_df,
    group_col="PostalCode",
    value_col="Margin",
    group_a=postal_code_a,
    group_b=postal_code_b
)

print("\nHypothesis 3 - Margin by Postal Codes")
print("P-value:", margin_p_value)
print("Decision:", decision(margin_p_value))


# ==============================
# HYPOTHESIS 4
# H0: No significant risk difference between women and men
# KPI: Claim Frequency
# ==============================

gender_df = df[
    df["Gender"].isin(["Male", "Female"])
]

gender_p_value = chi_square_test(
    data=gender_df,
    group_col="Gender",
    target_col="ClaimOccurred"
)

print("\nHypothesis 4 - Gender")
print("P-value:", gender_p_value)
print("Decision:", decision(gender_p_value))


# ==============================
# RESULTS SUMMARY TABLE
# ==============================

results = pd.DataFrame({
    "Hypothesis": [
        "Risk Difference Across Provinces",
        "Risk Difference Between Postal Codes",
        "Margin Difference Between Postal Codes",
        "Risk Difference Between Genders"
    ],
    "Test Used": [
        "Chi-Square Test",
        "Chi-Square Test",
        "T-Test",
        "Chi-Square Test"
    ],
    "P-Value": [
        province_p_value,
        postal_p_value,
        margin_p_value,
        gender_p_value
    ],
    "Decision": [
        decision(province_p_value),
        decision(postal_p_value),
        decision(margin_p_value),
        decision(gender_p_value)
    ]
})

results    

C:\Users\hp\AppData\Local\Temp\ipykernel_17752\2458747320.py:11: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")


Hypothesis 1 - Provinces
P-value: 5.925510718204678e-19
Decision: Reject H0

Hypothesis 2 - Postal Codes
P-value: 0.05788217295685014
Decision: Fail to Reject H0

Hypothesis 3 - Margin by Postal Codes
P-value: 0.24446241842452013
Decision: Fail to Reject H0

Hypothesis 4 - Gender
P-value: 0.9514644755420456
Decision: Fail to Reject H0


,Hypothesis,Test Used,P-Value,Decision
0,Risk Difference Across Provinces,Chi-Square Test,5.925511e-19,Reject H0
1,Risk Difference Between Postal Codes,Chi-Square Test,5.788217e-02,Fail to Reject H0
2,Margin Difference Between Postal Codes,T-Test,2.444624e-01,Fail to Reject H0
3,Risk Difference Between Genders,Chi-Square Test,9.514645e-01,Fail to Reject H0
